# Budgerigar：同音完整复读 episode 审计

目标 token 与输入 token 完全相同，只在输入结束后加入 160–280 ms 思考间隔。此阶段不转换音色。

In [ ]:
#@title 1. 可恢复地更新项目
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib,shutil,datetime
repo=Path(REPO_DIR)
if not (repo/'.git').is_dir():
    subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else:
    pull=subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],text=True,capture_output=True)
    print(pull.stdout,pull.stderr)
    if pull.returncode:
        backup=repo.with_name(f'Budgerigar_backup_{datetime.datetime.now():%H%M%S}')
        print('pull 失败；保留临时仓库到:',backup)
        shutil.move(str(repo),str(backup))
        subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train]'],check=True)
sys.path.insert(0,REPO_DIR)
for name in [k for k in list(sys.modules) if k=='budgerigar' or k.startswith('budgerigar.')]: del sys.modules[name]
importlib.invalidate_caches()
commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('commit:',commit)

In [ ]:
#@title 2. Drive 与 codec manifest
from google.colab import drive
drive.mount('/content/drive')
WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar')
CODEC_FINGERPRINT='e47d29bb8ba86e3e' #@param {type:'string'}
CODEC_MANIFEST=WORK_ROOT/'manifests'/f'cmu_arctic.encodec.{CODEC_FINGERPRINT}.jsonl'
assert CODEC_MANIFEST.is_file(),CODEC_MANIFEST

In [ ]:
#@title 3. 全量 manifest + 64 条均匀 payload 审计
from budgerigar.codec_copy_data import audit_codec_copy_manifest
audit=audit_codec_copy_manifest(CODEC_MANIFEST,sample_payloads=64)
print(audit)
assert audit['records']==2263 and audit['manifest_missing_required']==0
assert audit['manifest_fingerprints']==audit['payload_fingerprints']==[CODEC_FINGERPRINT]
assert len(audit['codebooks'])==1 and audit['token_min']>=0

In [ ]:
#@title 4. 连续时间轴检查
import torch
from budgerigar.codec_copy_data import CodecCopyEpisodeDataset
preview=CodecCopyEpisodeDataset(CODEC_MANIFEST,'train',thinking_ms=(160,280),max_records=8,preload=True)
for index in range(len(preview)):
    inputs,targets,voice,meta=preview[index]
    source=inputs[:meta['source_frames']]
    repeated=targets[meta['repeat_start']:]
    assert torch.equal(source,repeated)
    assert not voice[:meta['repeat_start']].any()
    print(meta,'exact_token_copy=',torch.equal(source,repeated))